In [50]:
import sys 
from pathlib import Path 

import pandas as pd
import numpy as np

import calendar
from datetime import datetime
from datetime import date

import plotly.graph_objects as go


# Resolve path to project root
root_path = Path.cwd().parents[1]
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from src.queries.smhi import *

## Data query import

- [ ] fix import for multiple meteo stations

In [ ]:
params_meteo={
    'version' : '1.0',
    'parameter' : ['1', '4', '6', '7', '39'],    # See here for more details https://opendata.smhi.se/metobs/resources/parameter
    'station' : '53530',    # This is the ID of the meteorolical station, which can be found here https://www.smhi.se/data/hitta-data-for-en-plats/ladda-ner-vaderobservationer/precipitationHourlySum
    'period' : 'corrected-archive',
    'data' : 'csv'
}

params_meteo_latest={
    'version' : '1.0',
    'parameter' : ['1', '4', '6', '7', '39'],    # See here for more details https://opendata.smhi.se/metobs/resources/parameter
    'station' : '53530',    # This is the ID of the meteorolical station, which can be found here https://www.smhi.se/data/hitta-data-for-en-plats/ladda-ner-vaderobservationer/precipitationHourlySum
    'period' : 'latest-months',
    'data'  : 'json'
}

params_hydro={
    'version' : 'latest',
    'parameter' : ['1', '3'],    # See here for more details https://opendata.smhi.se/metobs/resources/parameter
    'station' : ['2176', '2128', '2372',],     # This is the ID of the hydrological station, which can be found here https://www.smhi.se/data/hitta-data-for-en-plats/ladda-ner-observationer-fran-sjoar-och-vattendrag/waterLevel
    'period' : 'corrected-archive',
}

df_metops = SMHI_Metops(params_meteo)
df_metops_latest = SMHI_Metops(params_meteo_latest)
df_metops = pd.merge(df_metops, df_metops_latest, how='outer')

df_hydrops = SMHI_Hydroops(params_hydro)


# Transform df_hydrops from long format into wide format
df_hydrops_wide = df_hydrops.pivot_table(
    index='DateTime', 
    columns='StationName', 
    values=['Vattenföring (Dygn) [m³/s]', 'Vattenstånd [cm]'],
    aggfunc='mean'
)

# Flatten the multi-level df hydrops
df_hydrops_wide.columns = [f"{metric}_{station}" for metric, station in df_hydrops_wide.columns]
df_hydrops_wide = df_hydrops_wide.reset_index()



Request SMHI Open Data Meteorological Observations.
Request SMHI Open Data Meteorological Observations.
Request SMHI Open Data Hydrological Observations.


### Data cleaning

This part needs manual input, e.g. paramter names to be treated, as a fully automatic data cleaning may result in unwanted data loss.

In [25]:
# Clean the data due to anomalies
cols_to_clean = ['Vattenstånd [cm]_RINGSJÖDAMMEN ÖVRE']

for col in cols_to_clean:
    # Calculate the dynamic threshold
    rolling_mean = df_hydrops_wide[col].rolling(window=30, center=True).mean()
    rolling_std = df_hydrops_wide[col].rolling(window=30, center=True).std()

    # Flag anomalies 
    local_lower = rolling_mean - (3 * rolling_std)
    df_hydrops_wide.loc[df_hydrops_wide[col] < local_lower, col] = np.nan

    df_hydrops_wide.loc[df_hydrops_wide[col] == 0, col] = np.nan

    max_lower = 0.8 * df_hydrops_wide[col].mean()
    df_hydrops_wide.loc[df_hydrops_wide[col] < max_lower, col] = np.nan

## Plot creation

### Meteorology

#### Average Climate

In [43]:
df_tmp = df_metops.set_index('DateTime')

station_name = 'Hörby A (53530)'

# Resample to Monthly sums/means
df_tmp = df_tmp.resample('MS').agg({
    'Lufttemperatur': 'mean',
    'Nederbördsmängd': 'sum'
})

# Create average data
df_tmp = df_tmp.groupby(df_tmp.index.month).mean()


fig = go.Figure()

fig.add_trace(
    go.Bar(
        x = df_tmp.index,
        y = df_tmp['Nederbördsmängd'],
        name='Precipitation', 
        yaxis='y'
    )
)
fig.add_trace(
    go.Scatter(
        x = df_tmp.index, 
        y = df_tmp['Lufttemperatur'], 
        name = 'Airtemperature',
        yaxis='y2',
    )
)

fig.update_layout(
    template='simple_white',
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(
        #title='DateTime',
        tickvals = df_tmp.index,
        ticktext=calendar.month_name[1:]
    ),
    yaxis=dict(
        title='Precipitation [mm/month]',
        side='left', 
        range=[0, 85],
    ),
    yaxis2=dict(
        title='Airtemperature [°C]',
        side='right', 
        overlaying='y',
        range=[-2, 20]
    ), 
    legend=dict(
        xanchor='left', 
        x = 0.01, 
        yanchor='top', 
        y = 0.99
    )
)

# format the caption  
caption = f"Average climate at {station_name}"

## save caption
with open("../../fig/html/meteo_avg_rin_caption.md", "w") as f:
    f.write(caption)

fig.write_html("../../fig/html/meteo_avg_rin.html")

#### Current Climate

In [ ]:
current_year = datetime.today().year

df_tmp = df_metops[df_metops['DateTime'].dt.year == current_year].set_index('DateTime')
station_name = 'Hörby A (53530)'

month_names = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December",
}

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x = df_tmp.resample('ME').sum()['Nederbördsmängd'].index.month.map(month_names),
        y = df_tmp.resample('ME').sum()['Nederbördsmängd'],
        name = 'Precipitation',
        yaxis='y'
    )
)
fig.add_trace(
    go.Scatter(
        x = df_tmp.resample('ME').mean()['Lufttemperatur'].index.month.map(month_names),
        y = df_tmp.resample('ME').mean()['Lufttemperatur'],
        name = 'Airtemperature', 
        yaxis = 'y2'
    )
)

fig.update_layout(
    template='simple_white',
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(
        #title='Month', 
    ),
    yaxis=dict(
        title='Precipitation [mm/month]', 
        side='left',
        range=[0, 85],
    ),
    yaxis2=dict(
        title='Airtemperature [°C]', 
        side='right', 
        overlaying='y',
        range=[-2, 20]
    ),
    legend=dict(
        xanchor='left', 
        x = 0.01, 
        yanchor='top', 
        y = 0.99
    )
)

# format the caption  
caption = f"Current climate at {station_name}"

## save caption
with open("../../fig/html/meteo_current_rin_caption.md", "w") as f:
    f.write(caption)

fig.write_html("../../fig/html/meteo_current_rin.html")


### Hydrology

In [41]:
fig = go.Figure()

waterlevel_column = 'Vattenstånd [cm]_RINGSJÖDAMMEN ÖVRE'
station_name = df_hydrops_wide[waterlevel_column].name.split("_", 1)[1].title()

fig.add_trace(
    go.Scatter(
        x = df_hydrops_wide['DateTime'], 
        y = df_hydrops_wide[waterlevel_column] / 100,
        name = 'Waterlevel'
    )
)

fig.update_layout(
    template='simple_white', 
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(
        title='Timestamp'
    ),
    yaxis=dict(
        title='Waterlevel [masl]'
    ), 
)

# format the caption  
caption = f"Daily waterlevel at {station_name}"

## save caption
with open("../../fig/html/waterlevel_rin_caption.md", "w") as f:
    f.write(caption)

fig.write_html("../../fig/html/waterlevel_rin.html")